# AuditQuant — Fine-tune CodeT5 for Solidity Vulnerability Repair

This notebook fine-tunes **Salesforce/codet5-base** (220M params) on ~60 Solidity
vulnerable → fixed code pairs sourced from the **SWC Registry** and **DeFiVulnLabs**.

**Runtime**: ~15-20 min on a free T4 GPU (15 epochs).  
**Output**: A trained model you download and place at `backend/remediation/models/codet5-solidity-repair/`.

## Setup
1. Go to **Runtime → Change runtime type → T4 GPU** (free tier is fine)
2. Run all cells in order
3. Download the model zip at the end

## 1. Install dependencies

In [ ]:
!pip install -q "transformers==4.44.2" accelerate sentencepiece torch
# After running this cell, restart the runtime: Runtime → Restart session
# Then skip this cell and continue from Cell 2 onward.

## 2. Upload training data

Run the cell below, then **upload** these two files from your project when prompted:
- `backend/remediation/training/data/train.jsonl`
- `backend/remediation/training/data/eval.jsonl`

In [ ]:
from google.colab import files
import os

os.makedirs("data", exist_ok=True)

print("Upload train.jsonl and eval.jsonl (select both at once):")
uploaded = files.upload()

for name, content in uploaded.items():
    dest = os.path.join("data", name)
    with open(dest, "wb") as f:
        f.write(content)
    print(f"  Saved {dest} ({len(content)} bytes)")

## 3. Verify data

In [ ]:
import json

for split in ["train", "eval"]:
    path = f"data/{split}.jsonl"
    with open(path) as f:
        lines = [json.loads(l) for l in f if l.strip()]
    print(f"{split}: {len(lines)} examples")
    # Show first example
    ex = lines[0]
    print(f"  Input (first 120 chars):  {ex['input'][:120]}...")
    print(f"  Target (first 120 chars): {ex['target'][:120]}...")
    print()

## 4. Dataset & Model setup

In [ ]:
import json
import torch
from pathlib import Path
from torch.utils.data import Dataset
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)

BASE_MODEL = "Salesforce/codet5-base"
OUTPUT_DIR = "codet5-solidity-repair"
MAX_INPUT_LEN = 512
MAX_TARGET_LEN = 512


class SolidityRepairDataset(Dataset):
    def __init__(self, path, tokenizer, max_input_len, max_target_len):
        self.tokenizer = tokenizer
        self.max_input_len = max_input_len
        self.max_target_len = max_target_len
        self.examples = []
        with open(path) as f:
            for line in f:
                line = line.strip()
                if line:
                    self.examples.append(json.loads(line))
        print(f"Loaded {len(self.examples)} examples from {path}")

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        ex = self.examples[idx]
        model_inputs = self.tokenizer(
            ex["input"],
            max_length=self.max_input_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        labels = self.tokenizer(
            ex["target"],
            max_length=self.max_target_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        label_ids = labels["input_ids"].squeeze()
        label_ids[label_ids == self.tokenizer.pad_token_id] = -100
        return {
            "input_ids": model_inputs["input_ids"].squeeze(),
            "attention_mask": model_inputs["attention_mask"].squeeze(),
            "labels": label_ids,
        }


print(f"Loading tokenizer & model: {BASE_MODEL}")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
model = AutoModelForSeq2SeqLM.from_pretrained(BASE_MODEL)

train_dataset = SolidityRepairDataset("data/train.jsonl", tokenizer, MAX_INPUT_LEN, MAX_TARGET_LEN)
eval_dataset = SolidityRepairDataset("data/eval.jsonl", tokenizer, MAX_INPUT_LEN, MAX_TARGET_LEN)

print(f"\nDevice: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"Model params: {sum(p.numel() for p in model.parameters()) / 1e6:.0f}M")

## 5. Train

Hyperparameters (from the plan):
- **Epochs**: 15
- **Batch size**: 4 (effective 16 with grad accum)
- **Learning rate**: 5e-5 with linear warmup
- **FP16**: enabled for T4 speed-up
- **Best model**: saved by lowest eval loss

In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=15,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=5e-5,
    warmup_ratio=0.1,
    weight_decay=0.01,
    fp16=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    predict_with_generate=True,
    logging_steps=5,
    report_to="none",
    remove_unused_columns=False,
)

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding="longest",
    label_pad_token_id=-100,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
    tokenizer=tokenizer,
)

print("Starting training...")
trainer.train()
print("Training complete!")

## 6. Save final model

In [ ]:
FINAL_DIR = "codet5-solidity-repair-final"
trainer.save_model(FINAL_DIR)
tokenizer.save_pretrained(FINAL_DIR)

import os
print(f"\nSaved model to {FINAL_DIR}/")
for f in sorted(os.listdir(FINAL_DIR)):
    size = os.path.getsize(os.path.join(FINAL_DIR, f))
    print(f"  {f:40s} {size / 1024 / 1024:8.1f} MB")

## 7. Quick test — generate a patch

In [ ]:
test_input = """fix SWC-107 Reentrancy:
function withdraw(uint amount) public {
    require(balances[msg.sender] >= amount);
    (bool ok,) = msg.sender.call{value: amount}("");
    require(ok);
    balances[msg.sender] -= amount;
}"""

inputs = tokenizer(test_input, return_tensors="pt", truncation=True, max_length=512).to(model.device)
output_ids = model.generate(**inputs, max_length=512, num_beams=4, early_stopping=True)
result = tokenizer.decode(output_ids[0], skip_special_tokens=True)

print("=== Input (vulnerable) ===")
print(test_input)
print("\n=== Output (patched) ===")
print(result)

## 8. Download the trained model

This zips the model and triggers a browser download.  
After downloading, **unzip** into your project:

```bash
unzip codet5-solidity-repair-final.zip -d backend/remediation/models/codet5-solidity-repair/
```

The pipeline will automatically detect the weight files and use the fine-tuned model.

In [ ]:
import shutil
from google.colab import files

zip_path = shutil.make_archive("codet5-solidity-repair-final", "zip", ".", FINAL_DIR)
print(f"Created {zip_path} ({os.path.getsize(zip_path) / 1024 / 1024:.0f} MB)")
print("Downloading...")
files.download(zip_path)

---

## Done!

After unzipping the model into `backend/remediation/models/codet5-solidity-repair/`, the AuditQuant pipeline will automatically detect and use the fine-tuned model for vulnerability remediation.